# 02 - Filtragem Colaborativa com KNN

## Objetivo

Implementar e treinar o modelo KNN para filtragem colaborativa, com fundamentação matemática e justificativa da abordagem (user-based vs item-based).

## 1. Setup

In [16]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import joblib

## 2. Carregamento do dataset tratado

In [17]:
dataset_path = Path("../dataset/dataset.csv")

df = pd.read_csv(dataset_path)

df.head()

,user_id,item_id,ranking,valor_medio_ingresso,tipo_evento
0,1,UM MALLANDRO NA QUARENTENA COM SERGIO MALLANDRO,1,120.0,Show Stand-Up
1,2,DOMINGUINHOS - ISSO AQUI TÁ BOM DEMAIS,1,50.0,Show Musical
2,3,A NOVIÇA REBELDE,1,225.0,Espetáculo Musical
3,4,MUSICAL HAIRSPRAY,1,350.0,Espetáculo Musical
4,5,A NOVIÇA REBELDE,1,125.0,Espetáculo Musical


In [18]:
print("Dimensão do dataset:", df.shape)
print("Usuários únicos:", df["user_id"].nunique())
print("Itens únicos:", df["item_id"].nunique())
print("Ranking máximo:", df["ranking"].max())
print("Ranking mínimo:", df["ranking"].min())

Dimensão do dataset: (109916, 5)
Usuários únicos: 68499
Itens únicos: 449
Ranking máximo: 5
Ranking mínimo: 1


O dataset utilizado neste notebook foi gerado na etapa anterior de análise e preparação dos dados. Ele contém as colunas `user_id`, `item_id` e `ranking`, que representam respectivamente o usuário, o espetáculo e a força da interação entre eles.

Como a base original não possui avaliações explícitas, o `ranking` foi construído como feedback implícito, representando a quantidade de dias distintos em que o usuário interagiu com determinado espetáculo, limitado ao valor máximo de 5.

## 3. Fundamentação Teórica

### 3.1. Lógica do algoritmo

O KNN, ou *K-Nearest Neighbors*, é um algoritmo baseado em proximidade. Em sistemas de recomendação, ele é usado para encontrar usuários ou itens semelhantes a partir de uma matriz de interações.

No contexto deste projeto, a matriz usuário-item representa as interações entre usuários e espetáculos da Aventura. Cada linha representa um usuário, cada coluna representa um espetáculo e cada célula contém o valor de `ranking`, que indica a força da interação entre aquele usuário e aquele espetáculo.

A filtragem colaborativa com KNN funciona em três etapas principais:

1. **Cálculo de similaridade:** o algoritmo mede o quão parecidos são dois usuários ou dois itens com base nos padrões de interação.
2. **Seleção dos k vizinhos mais próximos:** após calcular as similaridades, são selecionados os `k` usuários ou itens mais semelhantes.
3. **Geração da recomendação:** o modelo utiliza os vizinhos mais próximos para estimar quais itens podem ser relevantes para o usuário.

Neste notebook, o KNN será utilizado em uma abordagem **item-based**, ou seja, o modelo buscará espetáculos semelhantes entre si. Dessa forma, se um usuário interagiu com determinado espetáculo, o sistema poderá recomendar outros espetáculos parecidos com ele.

---

### 3.2. Métricas de similaridade

Para que o KNN consiga encontrar os vizinhos mais próximos, é necessário definir uma métrica de similaridade. Essa métrica determina como o modelo calcula a proximidade entre dois usuários ou dois itens.

As principais métricas utilizadas em sistemas de recomendação são:

#### Similaridade do Cosseno

A similaridade do cosseno mede o ângulo entre dois vetores. No caso de recomendação item-based, cada item é representado por um vetor contendo as interações dos usuários com aquele item.

A fórmula é:

$$
\text{sim}(A,B) = \frac{\vec{A} \cdot \vec{B}}{||\vec{A}|| \cdot ||\vec{B}||}
$$

Onde:

- $\vec{A}$ representa o vetor de interações do item A;
- $\vec{B}$ representa o vetor de interações do item B;
- $\vec{A} \cdot \vec{B}$ representa o produto escalar entre os dois vetores;
- $||\vec{A}||$ representa a norma, ou magnitude, do vetor A;
- $||\vec{B}||$ representa a norma, ou magnitude, do vetor B.

Quanto mais próximo de 1 for o resultado, mais semelhantes são os itens. Quanto mais próximo de 0, menor é a similaridade entre eles.

Neste projeto, a similaridade do cosseno é adequada porque a matriz usuário-item é esparsa, ou seja, possui muitos valores iguais a 0. A métrica do cosseno permite comparar padrões de interação mesmo quando os usuários interagem com poucos itens do catálogo.

#### Correlação de Pearson

A correlação de Pearson mede a relação linear entre dois vetores, considerando a variação em torno da média. Ela é útil quando se deseja reduzir o impacto de usuários que tendem a avaliar tudo muito alto ou muito baixo.

A fórmula geral é:

$$
\text{sim}(u,v) = \frac{\sum_i (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_i (r_{ui} - \bar{r}_u)^2} \sqrt{\sum_i (r_{vi} - \bar{r}_v)^2}}
$$

Onde:

- $r_{ui}$ é a interação do usuário $u$ com o item $i$;
- $r_{vi}$ é a interação do usuário $v$ com o item $i$;
- $\bar{r}_u$ é a média das interações do usuário $u$;
- $\bar{r}_v$ é a média das interações do usuário $v$.

Apesar de ser útil em bases com avaliações explícitas, como notas de 1 a 5, ela pode ser menos adequada neste caso, pois o `ranking` representa feedback implícito derivado de compras, e não avaliações subjetivas dadas pelos usuários.

#### MSD — Mean Squared Difference

A métrica MSD, ou diferença quadrática média, calcula a distância entre dois vetores com base nas diferenças entre suas interações.

A ideia é medir o quanto dois usuários ou itens se diferenciam em suas interações. Quanto menor a diferença quadrática média, mais semelhantes eles são.

A fórmula pode ser representada como:

$$
MSD(u,v) = \frac{1}{|I_{uv}|} \sum_{i \in I_{uv}} (r_{ui} - r_{vi})^2
$$

Onde:

- $I_{uv}$ representa o conjunto de itens em comum entre os usuários $u$ e $v$;
- $r_{ui}$ é a interação do usuário $u$ com o item $i$;
- $r_{vi}$ é a interação do usuário $v$ com o item $i$.

Essa métrica é simples, mas pode ser mais sensível à escala dos valores de interação.

---

### 3.3. Fórmula de predição

Em abordagens de KNN para recomendação, a predição de uma interação pode ser calculada a partir da média ponderada das interações dos vizinhos mais próximos.

Uma formulação comum, especialmente em modelos como KNNWithMeans, é:

$$
\hat{r}_{ui} = \mu_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u,v) \cdot (r_{vi} - \mu_v)}{\sum_{v \in N_k(u)} |\text{sim}(u,v)|}
$$

Onde:

- $\hat{r}_{ui}$ é a interação prevista do usuário $u$ com o item $i$;
- $\mu_u$ é a média das interações do usuário $u$;
- $N_k(u)$ representa o conjunto dos $k$ vizinhos mais próximos do usuário $u$;
- $\text{sim}(u,v)$ é a similaridade entre o usuário $u$ e o vizinho $v$;
- $r_{vi}$ é a interação real do vizinho $v$ com o item $i$;
- $\mu_v$ é a média das interações do vizinho $v$;
- o denominador $\sum_{v \in N_k(u)} |\text{sim}(u,v)|$ normaliza o cálculo pela soma das similaridades absolutas.

A fórmula ajusta a predição considerando a média do usuário e o comportamento dos vizinhos. Dessa forma, vizinhos mais semelhantes têm maior peso na estimativa.

No caso deste notebook, como será utilizada uma abordagem item-based com `NearestNeighbors`, o foco inicial não será prever diretamente um rating numérico para todos os pares usuário-item, mas sim identificar espetáculos mais semelhantes aos que o usuário já consumiu e gerar recomendações a partir dessas similaridades.

---

### 3.4. Justificativa da métrica escolhida

A métrica escolhida para este modelo foi a **similaridade do cosseno**.

Essa escolha se justifica porque:

- a matriz usuário-item possui alta esparsidade;
- o objetivo é comparar padrões de interação entre espetáculos;
- o cosseno é adequado para comparar vetores mesmo quando há muitos valores 0;
- a escala do `ranking` já foi limitada entre 1 e 5, reduzindo distorções;
- a abordagem facilita a identificação de itens semelhantes com base nos usuários que interagiram com eles.

Portanto, a similaridade do cosseno é uma escolha coerente para o primeiro modelo KNN de filtragem colaborativa aplicado ao sistema de recomendação da Aventura.

## 4. User-based vs Item-based

A filtragem colaborativa com KNN pode ser implementada de duas formas principais: baseada em usuários ou baseada em itens.

Na abordagem **user-based**, o modelo compara usuários entre si. A ideia é recomendar itens consumidos por usuários parecidos com o usuário alvo. Por exemplo, se dois usuários consumiram espetáculos semelhantes, os itens consumidos por um deles podem ser recomendados ao outro.

Na abordagem **item-based**, o modelo compara itens entre si. A ideia é recomendar itens semelhantes àqueles que o usuário já consumiu. Por exemplo, se muitos usuários que compraram um espetáculo também compraram outro, esses dois espetáculos podem ser considerados semelhantes.

Para este projeto, foi escolhida a abordagem **item-based**.

Essa decisão foi tomada porque o dataset possui uma quantidade muito maior de usuários do que de itens. Na etapa de preparação dos dados, a matriz usuário-item apresentou aproximadamente 68 mil usuários e 571 espetáculos. Dessa forma, é mais eficiente comparar os espetáculos entre si do que comparar todos os usuários.

Além disso, no contexto da Aventura, a abordagem item-based é mais interpretável, pois permite recomendar espetáculos semelhantes aos que o usuário já consumiu. Isso facilita a explicação da recomendação e está alinhado ao objetivo de aumentar a descoberta de espetáculos no portal.

## 5. Criação da matriz usuário-item

Nesta etapa, o dataset tratado é transformado em uma matriz usuário-item. Cada linha representa um usuário, cada coluna representa um espetáculo e cada célula contém o valor do `ranking`.

Quando não existe interação entre um usuário e um espetáculo, o valor é preenchido com 0. Essa estrutura é necessária para que o KNN consiga comparar padrões de interação entre os espetáculos.

In [19]:
matriz_usuario_item = df.pivot_table(
    index="user_id",
    columns="item_id",
    values="ranking",
    fill_value=0
)

print("Formato da matriz usuário-item:", matriz_usuario_item.shape)

matriz_usuario_item.head()

Formato da matriz usuário-item: (68499, 449)


item_id,"""DUDA BRACK - CACO DE VIDRO""","""EU NÃO DIGO É NADA "" COM GLAUBER CUNHA","""MÚSICA, LETRA E DANÇA"" BABI XAVIER, CAROL FAZU E HELGA NEMETIK CANTAM MARINA","""TACY - RELICÁRIO DE CÁSSIA ELLER""","""TÁ EMBAÇADO"" COM FABIO RABIN",(1) LONGA JORNADA NOITE ADENTRO,4 AMIGOS,A BELLA ITÁLIA - A BELA E OS TENORES - ESPECIAL ANDREA BOCELLI.,"A COR PÚRPURA, O MUSICAL",A CURA DA DEPRESSÃO – DIOGO DEFANTE,...,ZABELÊ - “AUÊ” SESSION`S #TEATRORIACHUELO5ANOS,ZAQUIM,ZECA BALEIRO – JOSÉ,ZÉ IBARRA,ZÉ IBARRA E DORA MORELENBAUM NO SOMAMOS FESTIVAL,ZÉLIA DUNCAN- SEM TIRAR OS OLHOS DO MUNDO,ÁUREA MARTINS - SENHORA DAS FOLHAS,ÁUREA MARTINS – LANÇAMENTO DO DISCO “SENHORA DAS FOLHAS”,ÓPERA DE PEQUIM - CHINA NATIONAL PEKING OPERA COMPANY,“PARA LENNON & MCCARTNEY – OS BEATLES E O CLUBE DA ESQUINA”
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Preparação da matriz item-usuário

Como a abordagem escolhida foi item-based, a matriz usuário-item precisa ser transposta. Assim, cada linha passa a representar um espetáculo e cada coluna representa um usuário.

Essa estrutura permite que o KNN compare espetáculos entre si com base nos usuários que interagiram com eles.

In [20]:
matriz_item_usuario = matriz_usuario_item.T

print("Formato da matriz item-usuário:", matriz_item_usuario.shape)

matriz_item_usuario.head()

Formato da matriz item-usuário: (449, 68499)


user_id,1,2,3,4,5,6,7,8,9,10,...,68554,68555,68556,68557,68558,68559,68560,68561,68562,68563
item_id,,,,,,,,,,,,,,,,,,,,,
"""DUDA BRACK - CACO DE VIDRO""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""EU NÃO DIGO É NADA "" COM GLAUBER CUNHA",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""MÚSICA, LETRA E DANÇA"" BABI XAVIER, CAROL FAZU E HELGA NEMETIK CANTAM MARINA",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""TACY - RELICÁRIO DE CÁSSIA ELLER""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""TÁ EMBAÇADO"" COM FABIO RABIN",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Conversão para matriz esparsa

A matriz item-usuário possui muitos valores iguais a 0, pois cada usuário interage com apenas uma pequena parte do catálogo. Para tornar o processamento mais eficiente, a matriz é convertida para o formato esparso.

In [21]:
matriz_esparsa = csr_matrix(matriz_item_usuario.values)

matriz_esparsa

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 109916 stored elements and shape (449, 68499)>

### 7.1 Definição dos hiperparâmetros

O modelo foi configurado com `n_neighbors = 6`. Esse valor foi escolhido porque, ao buscar os vizinhos mais próximos de um espetáculo, o primeiro vizinho retornado tende a ser o próprio espetáculo. Portanto, são buscados 6 vizinhos para que, após remover o próprio item, restem 5 espetáculos similares para recomendação.

A métrica escolhida foi `cosine`, pois a matriz item-usuário é esparsa e a similaridade do cosseno é adequada para comparar padrões de interação em vetores com muitos valores 0.

O algoritmo utilizado foi `brute`, pois calcula diretamente as distâncias entre os itens. Como a base possui uma quantidade relativamente pequena de espetáculos em comparação com usuários, essa abordagem é viável para este primeiro modelo.

## 8. Treinamento do modelo KNN

Nesta etapa, o modelo KNN é treinado a partir da matriz item-usuário em formato esparso.

Como a abordagem escolhida foi item-based, cada linha da matriz representa um espetáculo e cada coluna representa um usuário. Assim, o modelo aprende a identificar quais espetáculos possuem padrões de interação semelhantes.

A métrica utilizada será a distância do cosseno. No `NearestNeighbors`, valores menores de distância indicam maior proximidade entre os itens. Como a similaridade do cosseno é dada por `1 - distância`, itens com distância próxima de 0 são considerados mais semelhantes.

In [22]:
modelo_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

modelo_knn.fit(matriz_esparsa)

print("Modelo KNN treinado com sucesso.")

Modelo KNN treinado com sucesso.


## 9. Função para buscar espetáculos similares

Após o treinamento do KNN, foi criada uma função para buscar os espetáculos mais semelhantes a um espetáculo de referência.

A função recebe o nome de um espetáculo, localiza sua posição na matriz item-usuário e utiliza o modelo KNN para encontrar os vizinhos mais próximos. O primeiro vizinho retornado normalmente é o próprio espetáculo, por isso ele é removido da lista final.

In [23]:
def buscar_itens_similares(item_id, top_n=5):

    if item_id not in matriz_item_usuario.index:
        raise ValueError(f"O item '{item_id}' não foi encontrado na matriz.")

    item_index = matriz_item_usuario.index.get_loc(item_id)

    distancias, indices = modelo_knn.kneighbors(
        matriz_esparsa[item_index],
        n_neighbors=top_n + 1
    )

    resultados = []

    for distancia, indice in zip(distancias.flatten(), indices.flatten()):
        item_similar = matriz_item_usuario.index[indice]

        if item_similar != item_id:
            resultados.append({
                "item_id": item_similar,
                "distancia": distancia,
                "similaridade": 1 - distancia
            })

    return pd.DataFrame(resultados).head(top_n)

In [24]:
item_exemplo = matriz_item_usuario.index[0]

print("Item de referência:", item_exemplo)

buscar_itens_similares(item_exemplo, top_n=5)

Item de referência: "DUDA BRACK - CACO DE VIDRO"


,item_id,distancia,similaridade
0,SOMAMOS FESTIVAL – DUDA BRACK & CHICO CHICO - ...,0.905084,0.094916
1,SOMAMOS FESTIVAL – CHICO BROWN,0.941876,0.058124
2,NEY MATOGROSSO – VOZ E PIANO,0.952040,0.047960
3,ELBA RAMALHO – 40 ANOS,0.952542,0.047458
4,ANCESTRALIDADES EM CENA APRESENTA: DIVERSIDADE...,0.953959,0.046041


## 10. Função de recomendação para usuário

Nesta etapa, foi criada uma função para gerar recomendações para um usuário específico.

A lógica da função é:

1. identificar os espetáculos já consumidos pelo usuário;
2. buscar espetáculos similares a cada item consumido;
3. remover itens que o usuário já consumiu;
4. somar os scores de similaridade dos itens candidatos;
5. retornar os espetáculos com maior score de recomendação.

Essa abordagem permite recomendar itens semelhantes ao histórico do usuário, mantendo a lógica da filtragem colaborativa baseada em itens.

In [25]:
def recomendar_para_usuario(user_id, top_n=5):


    if user_id not in matriz_usuario_item.index:
        raise ValueError(f"O usuário '{user_id}' não foi encontrado na matriz.")

    interacoes_usuario = matriz_usuario_item.loc[user_id]

    itens_consumidos = interacoes_usuario[interacoes_usuario > 0].index.tolist()

    if len(itens_consumidos) == 0:
        return pd.DataFrame(columns=["item_id", "score_recomendacao"])

    recomendacoes = []

    for item in itens_consumidos:
        similares = buscar_itens_similares(item, top_n=top_n)

        for _, row in similares.iterrows():
            item_recomendado = row["item_id"]

            if item_recomendado not in itens_consumidos:
                recomendacoes.append({
                    "item_id": item_recomendado,
                    "score_recomendacao": row["similaridade"]
                })

    if len(recomendacoes) == 0:
        return pd.DataFrame(columns=["item_id", "score_recomendacao"])

    df_recomendacoes = pd.DataFrame(recomendacoes)

    df_recomendacoes = (
        df_recomendacoes
        .groupby("item_id", as_index=False)
        .agg(score_recomendacao=("score_recomendacao", "sum"))
        .sort_values("score_recomendacao", ascending=False)
        .head(top_n)
    )

    return df_recomendacoes

## 11. Teste com usuário de exemplo

Para validar a função de recomendação, foi selecionado um usuário com quantidade moderada de interações na base. Usuários com poucas interações podem gerar recomendações pouco informativas, enquanto usuários extremamente ativos podem representar comportamentos atípicos.

Por isso, foi escolhido um usuário com volume intermediário de consumo, permitindo observar melhor o funcionamento do modelo em um perfil mais representativo.

In [26]:
usuarios_ativos = (
    df.groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

usuarios_ativos.head()

user_id
47298    19
26889    19
16578    19
45809    19
64907    19
dtype: int64

In [29]:
usuarios_validos = usuarios_ativos[
    (usuarios_ativos >= 5) & (usuarios_ativos <= 30)
]

usuario_exemplo = usuarios_validos.index[0]

print("Usuário selecionado:", usuario_exemplo)
print("Quantidade de itens consumidos:", usuarios_validos.loc[usuario_exemplo])

print("\nItens já consumidos pelo usuário:")
display(
    df[df["user_id"] == usuario_exemplo][["item_id", "ranking"]]
    .sort_values("ranking", ascending=False)
    .head(10)
)

print("\nRecomendações geradas:")
recomendar_para_usuario(usuario_exemplo, top_n=5)

Usuário selecionado: 47298
Quantidade de itens consumidos: 19

Itens já consumidos pelo usuário:


,item_id,ranking
75471,PORTÁTIL,5
75469,NOEL ROSA: COISA NOSSA,2
75472,PRIMA FACIE COM DÉBORA FALABELLA,2
75466,KING KONG FRAN,2
75461,BRUNA LOUISE - ELA TÁ CORRENDO ATRÁS,1
75459,A NOVIÇA REBELDE,1
75460,AFONSO PADILHA - NINGUÉM SE IMPORTA,1
75465,IGOR GUIMARÃES - BENIGNISMO,1
75464,CONTO DE NATAL - UM BALLET DE DALAL ACHCAR,1
75462,CARLOTA / FOCUS DANÇA PIAZZOLLA,1



Recomendações geradas:


,item_id,score_recomendacao
32,NOSSA HISTÓRIA COM CHICO BUARQUE,0.451465
16,"ELIS, A MUSICAL",0.181752
45,VIVA O POVO BRASILEIRO [DE NAÊ À DAFÉ],0.177827
25,MACACOS,0.169634
43,UM FILME ARGENTINO - COM LETÍCIA COLIN E MICHE...,0.169100


## 12. Exportação do modelo

O modelo KNN treinado e as matrizes utilizadas foram salvos com `joblib`. Essa etapa permite reutilizar o modelo fora do notebook, por exemplo em scripts dentro da pasta `src/models` ou em uma futura função de inferência.

In [28]:
model_dir = Path("../src/models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(modelo_knn, model_dir / "knn_recommender_model.joblib")
joblib.dump(matriz_usuario_item, model_dir / "matriz_usuario_item.joblib")
joblib.dump(matriz_item_usuario, model_dir / "matriz_item_usuario.joblib")

print("Modelo e matrizes salvos em:", model_dir)

Modelo e matrizes salvos em: ..\src\models


## 13. Conclusões

Neste notebook, foi implementado um modelo inicial de recomendação utilizando filtragem colaborativa baseada em itens com KNN.

A abordagem item-based foi escolhida porque a base possui uma quantidade muito maior de usuários do que de itens, tornando mais viável comparar espetáculos entre si. Além disso, essa abordagem é mais interpretável para o contexto da Aventura, pois permite recomendar espetáculos semelhantes aos que o usuário já consumiu.

O modelo utilizou distância do cosseno para identificar itens próximos e foi capaz de gerar recomendações para usuários reais da base. Essa implementação representa a primeira etapa de modelagem do sistema de recomendação e servirá como base para comparações futuras com SVD ou Funk SVD.

Como próximos passos, recomenda-se realizar a divisão entre treino e teste, calcular métricas como RMSE e MAE, avaliar métricas de classificação como precisão, recall, F1-score e acurácia, e comparar o desempenho do KNN com modelos de fatoração de matrizes.